<a href="https://colab.research.google.com/github/Satishpk15/Machine-learning/blob/main/B2_Logistic_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# On Titanic Dataset

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Load the Titanic dataset (built into seaborn — no file download needed)
df = sns.load_dataset('titanic')
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


- Understand the data
- Cleaning
- One Hot Encoding
- Train test split
- Scaling
- ML model
- Evalutaiton


In [ ]:
print("Target distribution:")
print(df['survived'].value_counts())
print(f"\nSurvival rate: {df['survived'].mean():.1%}")
print(f"Death rate: {1 - df['survived'].mean():.1%}")

Target distribution:
survived
0    549
1    342
Name: count, dtype: int64

Survival rate: 38.4%
Death rate: 61.6%


In [ ]:
# Select features for our model
features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
target = 'survived'

# Create a clean working copy
data = df[features + [target]].copy()
print(f"Working dataset: {data.shape}")
print(f"\nMissing values:")
print(data.isnull().sum())

Working dataset: (891, 8)

Missing values:
pclass        0
sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
survived      0
dtype: int64


In [ ]:
data

,pclass,sex,age,sibsp,parch,fare,embarked,survived
0,3,male,22.0,1,0,7.2500,S,0
1,1,female,38.0,1,0,71.2833,C,1
2,3,female,26.0,0,0,7.9250,S,1
3,1,female,35.0,1,0,53.1000,S,1
4,3,male,35.0,0,0,8.0500,S,0
...,...,...,...,...,...,...,...,...
886,2,male,27.0,0,0,13.0000,S,0
887,1,female,19.0,0,0,30.0000,S,1
888,3,female,NaN,1,2,23.4500,S,0
889,1,male,26.0,0,0,30.0000,C,1


In [ ]:
# Age: fill with median (safe choice for skewed data)
data['age'] = data['age'].fillna(data['age'].median())

# Embarked: fill with mode (most common port)
data['embarked'] = data['embarked'].fillna(data['embarked'].mode()[0])

# Verify — no more missing values
print("Missing values after cleaning:")
print(data.isnull().sum())
print(f"\nDataset shape: {data.shape}")


Missing values after cleaning:
pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
survived    0
dtype: int64

Dataset shape: (891, 8)


In [ ]:
# One-hot encode categorical features
data_encoded = pd.get_dummies(data, columns=['sex', 'embarked'], drop_first=True)

print(f"Shape after encoding: {data_encoded.shape}")
print(f"\nColumns:")
print(data_encoded.columns.tolist())
data_encoded.head()


Shape after encoding: (891, 9)

Columns:
['pclass', 'age', 'sibsp', 'parch', 'fare', 'survived', 'sex_male', 'embarked_Q', 'embarked_S']


,pclass,age,sibsp,parch,fare,survived,sex_male,embarked_Q,embarked_S
0,3,22.0,1,0,7.2500,0,True,False,True
1,1,38.0,1,0,71.2833,1,False,False,False
2,3,26.0,0,0,7.9250,1,False,False,True
3,1,35.0,1,0,53.1000,1,False,False,True
4,3,35.0,0,0,8.0500,0,True,False,True


In [ ]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = data_encoded.drop(target, axis=1)
y = data_encoded[target]

# Split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} rows")
print(f"Test set:     {X_test.shape[0]} rows")
print(f"\nTraining survival rate: {y_train.mean():.1%}")
print(f"Test survival rate:     {y_test.mean():.1%}")


Training set: 712 rows
Test set:     179 rows

Training survival rate: 38.3%
Test survival rate:     38.5%


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on train, transform train
X_test_scaled = scaler.transform(X_test)          # only transform test (no fit!)

# Convert back to DataFrame for readability
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print("Scaled training data (first 3 rows):")
X_train_scaled.head(3)


Scaled training data (first 3 rows):


,pclass,age,sibsp,parch,fare,sex_male,embarked_Q,embarked_S
692,0.829568,-0.112078,-0.465084,-0.466183,0.513812,0.742427,-0.289333,0.611978
481,-0.370945,-0.112078,-0.465084,-0.466183,-0.662563,0.742427,-0.289333,0.611978
527,-1.571457,-0.112078,-0.465084,-0.466183,3.955399,0.742427,-0.289333,0.611978


In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train_scaled, y_train)

print("Model Trained")
print(f"\n Coefficients shape: {model.coef_.shape}")
print(f"Intercept: {model.intercept_[0]:.4f}")

Model Trained

 Coefficients shape: (1, 8)
Intercept: -0.6569


In [ ]:
probabilities = model.predict_proba(X_test_scaled)
probabilities

array([[0.93374384, 0.06625616],
       [0.95342876, 0.04657124],
       [0.84840028, 0.15159972],
       [0.96520361, 0.03479639],
       [0.31638207, 0.68361793],
       [0.55539745, 0.44460255],
       [0.24395899, 0.75604101],
       [0.67392797, 0.32607203],
       [0.65649221, 0.34350779],
       [0.84093315, 0.15906685],
       [0.83895356, 0.16104644],
       [0.91289994, 0.08710006],
       [0.41262519, 0.58737481],
       [0.77222146, 0.22777854],
       [0.53385836, 0.46614164],
       [0.82268674, 0.17731326],
       [0.61602697, 0.38397303],
       [0.91196548, 0.08803452],
       [0.87159212, 0.12840788],
       [0.23000015, 0.76999985],
       [0.91196548, 0.08803452],
       [0.22045118, 0.77954882],
       [0.91797063, 0.08202937],
       [0.56418625, 0.43581375],
       [0.91504438, 0.08495562],
       [0.03872078, 0.96127922],
       [0.83895356, 0.16104644],
       [0.73824308, 0.26175692],
       [0.8798457 , 0.1201543 ],
       [0.87421503, 0.12578497],
       [0.

In [ ]:
predicitons = model.predict(X_test_scaled)
predicitons

array([0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1,
       0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1,
       1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1,
       1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1,
       1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1,
       0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0,
       0, 1, 0])